# Chapter 4 — Reasoning with probabilities · Solution with comments

*Maths for AI, step by step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch04_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer starts with the reasoning on paper, then shows its equivalent in Python. For each calculation, the comments point out the group that serves as the denominator.

**Contents.** Data · The calculations of lessons 4.1 to 4.5 · Exercises 4.1 to 4.6

**Before each calculation**, ask yourself which set you are counting in: spam messages, flagged messages or received messages.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colours of the book: one colour = one role
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLEU, ORANGE = '#2a78d6', '#eb6834'   # “no spam missed”, normal distribution curve
from math import comb

## Data

The identifiers D4.1 and D4.2 are those of the book and of the CSV files. Variable and column names in the code are kept in French: `boite_20` and `boite_2` are the two mailboxes, `signalé` = flagged, `non signalé` = not flagged, `légitime` = legitimate, `durees` = processing times, `loi` = distribution, `manqué` = missed, `moyenne` = mean, `ecart_type` = standard deviation.

In [ ]:
# Chapter 4 data, copied from the book's CSV files (do not modify)
# Rows: spam, legitimate. Columns: flagged, not flagged.
LIGNES, COLONNES = ['spam', 'légitime'], ['signalé', 'non signalé']

# D4.1 — mailbox where 20 % of messages are spam (1 000 messages)
boite_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=LIGNES, columns=COLONNES)

# D4.2 — same filter, mailbox where 2 % of messages are spam (1 000 messages)
boite_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=LIGNES, columns=COLONNES)

# D4.3 — 1 000 processing times (ms), regenerated identically to the CSV file
durees = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

## The calculations of the lessons, in Python

These cells redo the calculations of lessons 4.1 to 4.5.

In [ ]:
# Lesson 4.1 — count, then divide by the right group (equation 4.1)
t = boite_20
n = t.values.sum()
print('total:', n, '| spam:', t.loc['spam'].sum(), '| flagged:', t['signalé'].sum())
print('P(spam) =', t.loc['spam'].sum() / n, '| P(flagged) =', t['signalé'].sum() / n,
      '| P(spam and flagged) =', t.loc['spam', 'signalé'] / n)
# Among the flagged messages: the denominator is the “signalé” column
print(f"spam among flagged messages: {t.loc['spam', 'signalé']} / {t['signalé'].sum()} "
      f"≈ {t.loc['spam', 'signalé'] / t['signalé'].sum():.2f}")

In [ ]:
# Lesson 4.2 — the same filter in a mailbox where spam is rare (equation 4.2)
d, f = 0.90, 0.05                   # flagged shares: of spam, of legitimate messages
for p, t in [(0.20, boite_20), (0.02, boite_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    reconstruit = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(reconstruit) == t.values).all()   # the table follows from the rates
    print(f'base rate {p:.0%}: {t.values.tolist()}  →  share of spam among flagged messages '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Lesson 4.3 — two conditional probabilities, two denominators (equation 4.3)
def conditionnelle(t, case, groupe):
    """Share of the cell (row, column) within the group: a row ('spam') or a column ('signalé')."""
    denominateur = t.loc[groupe].sum() if groupe in t.index else t[groupe].sum()
    return t.loc[case] / denominateur, denominateur

for nom, t in [('20 %', boite_20), ('2 %', boite_2)]:
    rappel, den_r = conditionnelle(t, ('spam', 'signalé'), 'spam')
    precision, den_p = conditionnelle(t, ('spam', 'signalé'), 'signalé')
    print(f'mailbox with {nom}: P(flagged | spam) = {rappel:.2f} (denominator {den_r}); '
          f'P(spam | flagged) = {precision:.2f} (denominator {den_p})')

# The same numbers with scikit-learn, from message-by-message labels (if it is installed)
try:
    from sklearn.metrics import precision_score, recall_score
    vrai = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = spam
    predit = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = flagged
    print(f'scikit-learn: recall = {recall_score(vrai, predit):.2f}, precision = {precision_score(vrai, predit):.2f}')
except ImportError:
    print('scikit-learn not installed: this calculation is optional.')

In [ ]:
# Lesson 4.4 — Bayes' theorem (equations 4.4 and 4.5) and the book's curve
def bayes(p, d=0.90, f=0.05):
    p_signale = d * p + f * (1 - p)                  # equation 4.5
    return d * p / p_signale                        # equation 4.4

for p in (0.20, 0.02):
    print(f'P(spam) = {p}: P(flagged) = {0.9 * p + 0.05 * (1 - p):.3f}, P(spam | flagged) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=SIGNALE, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Base rate: share of spam (%)'); plt.ylabel('P(spam | flagged)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lesson 4.5 — ten spam messages: binomial distribution (equation 4.7) and complementary event (equation 4.8)
n, p = 10, 0.1                        # we count over 10 SPAM messages; p = probability of being missed
loi = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0.9^10 =', round(0.9 ** 10, 4), '| at least one missed =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 to 4:', [round(v, 3) for v in loi[:5]], '| sum =', round(sum(loi), 6))
couleurs = [BLEU] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), loi, color=couleurs, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Number of missed spam messages out of 10 spam messages'); plt.ylabel('Probability'); plt.show()

# A glimpse of the normal distribution: processing times (D4.3)
m, s = durees.mean(), durees.std()
print(f'mean = {m:.2f} ms, standard deviation = {s:.2f} ms; within ±1 standard deviation: {np.mean(abs(durees - m) <= s):.1%}, '
      f'within ±2 standard deviations: {np.mean(abs(durees - m) <= 2 * s):.1%}')

## Exercise 4.1 — Reading a chart: twenty spam messages

*Concepts: binomial distribution, complementary event (lesson 4.5).* First read the figure in the book.

### a) to d) Solution with comments

Here we count over **20 spam messages**, not over received messages.

**a)** No spam missed: about 0.36. Exactly one: about 0.38.

**b)** The tallest bar is the one at 1: one missed spam is the most likely case.

**c)** $1 - 0{,}36 \approx 0{,}64$. With equation 4.8: $1 - 0{,}95^{20} \approx 0{,}6415$.

**d)** Each spam is half as likely to get through (5 % instead of 10 %), but there are twice as many spam messages (20 instead of 10). On average, the filter still misses one: $20 \times 0{,}05 = 1$. So the probability that at least one gets through stays close: 64 % versus 65 %.

In [ ]:
def binomiale(n, p, k):
    return comb(n, k) * p ** k * (1 - p) ** (n - k)

In [ ]:
# Check on the 20 spam messages of the exercise
if binomiale(20, 0.05, 0) is None:
    print('Complete the function.')
else:
    p0, p1 = binomiale(20, 0.05, 0), binomiale(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, at least one = {1 - p0:.4f} →',
          'correct' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'check again')

## Exercise 4.2 — A more cautious filter

*Concepts: conditional probability (lesson 4.3), Bayes' theorem (lesson 4.4), base rate (lesson 4.2).* The cautious filter flags 75 % of spam messages and 1 % of legitimate messages. Mailbox of 10 000 messages, 10 % of which are spam. First do questions a) to e) on paper.

### a) to c) Solution with comments

**a)** Start with the rows, that is, with what the messages are. 10 % of 10 000 gives **1 000 spam messages**, leaving **9 000 legitimate messages**. The filter flags 75 % of spam: 0.75 × 1 000 = **750**, so 250 spam messages get through. It flags 1 % of legitimate messages: 0.01 × 9 000 = **90** false alarms, and 8 910 legitimate messages get through.

| | Flagged | Not flagged | Total |
|:--|--:|--:|--:|
| Spam | 750 | 250 | 1 000 |
| Legitimate | 90 | 8 910 | 9 000 |
| **Total** | **840** | **9 160** | **10 000** |

**b)** $P(\text{signalé} \mid \text{spam})$: we count **in the spam row**, which serves as the denominator. $750 / 1\,000 = 0{,}75$. We get back the announced rate, which is reassuring. $P(\text{spam} \mid \text{signalé})$: we count **in the flagged column**. $750 / 840 \approx 0{,}893$. Same numerator, 750; different denominators, 1 000 and 840.

**c)** With equation 4.5: $P(\text{signalé}) = 0{,}75 \times 0{,}1 + 0{,}01 \times 0{,}9 = 0{,}075 + 0{,}009 = 0{,}084$, that is, the 840 flagged messages out of 10 000. With equation 4.4: $P(\text{spam} \mid \text{signalé}) = 0{,}075 / 0{,}084 \approx 0{,}893$. Bayes merely rewrites the counting in the table with proportions.

### a) The table in Python

In [ ]:
spam_signale = round(0.75 * 1000)          # 750: 75 % of the 1 000 spam messages
spam_non_signale = 1000 - spam_signale     # 250
legitime_signale = round(0.01 * 9000)      # 90 false alarms: 1 % of the 9 000 legitimate messages
legitime_non_signale = 9000 - legitime_signale   # 8 910

In [ ]:
# Check
cases = [spam_signale, spam_non_signale, legitime_signale, legitime_non_signale]
if None in cases:
    print('Complete the previous cell.')
else:
    prudent = pd.DataFrame([cases[:2], cases[2:]], index=LIGNES, columns=COLONNES)
    print(prudent)
    print('table:', 'correct' if cases == [750, 250, 90, 8910] else 'check again')

### b) The two conditional probabilities

In [ ]:
p_signale_si_spam = 750 / (750 + 250)   # denominator: the spam row
p_spam_si_signale = 750 / (750 + 90)    # denominator: the flagged column

In [ ]:
# Check
for nom, la_votre, vrai in [('P(flagged | spam)', p_signale_si_spam, 0.75),
                            ('P(spam | flagged)', p_spam_si_signale, 750 / 840)]:
    print(f'{nom}:', 'not yet completed' if la_votre is None else
          ('correct' if abs(la_votre - vrai) < 0.001 else 'check again'))

### c) to e) Bayes' theorem as a function

In [ ]:
def proba_spam_si_signale(p, d, f):
    p_signale = d * p + f * (1 - p)     # equation 4.5: the two cells of the “signalé” column
    return d * p / p_signale            # equation 4.4: the red cell divided by the column

In [ ]:
# Check of c), d) and e)
if proba_spam_si_signale(0.1, 0.75, 0.01) is None:
    print('Complete the function.')
else:
    cas = [('c) cautious, 10 % spam', (0.10, 0.75, 0.01), 0.8929),
           ('d) cautious, 1 % spam', (0.01, 0.75, 0.01), 0.4310),
           ('e) cautious, 2 % spam', (0.02, 0.75, 0.01), 0.6048),
           ('e) lessons filter, 2 % spam', (0.02, 0.90, 0.05), 0.2687)]
    for nom, args, attendu in cas:
        v = proba_spam_si_signale(*args)
        print(f'{nom}: {v:.4f} →', 'correct' if abs(v - attendu) < 0.0001 else 'check again')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, proba_spam_si_signale(P, 0.90, 0.05), color=SIGNALE, linewidth=2,
             label='lessons filter (90 %; 5 %)')
    plt.plot(100 * P, proba_spam_si_signale(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='cautious filter (75 %; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Base rate: share of spam (%)'); plt.ylabel('P(spam | flagged)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) and e) Solution with comments

**d)** Expected prediction: the probability **decreases**, since spam becomes rarer and false alarms weigh more (lesson 4.2). For 10 000 messages: 100 spam messages, 75 of them flagged; 9 900 legitimate messages, 99 of them flagged. In the flagged column: $75 / (75 + 99) = 75 / 174 \approx 0{,}431$. Fewer than one flag in two is real spam, even though the filter has not changed.

**e)** For the cautious filter in the 2 % mailbox: $\dfrac{0{,}75 \times 0{,}02}{0{,}75 \times 0{,}02 + 0{,}01 \times 0{,}98} = \dfrac{0{,}015}{0{,}0248} \approx 0{,}605$, versus 0.27 for the lessons filter. Comparing these two numbers means comparing the **precision** of the two filters: we count in the column of flagged messages. When the cautious filter flags a message, you can trust it more: it makes five times fewer false alarms. But precision is not enough to choose a filter. Count in the spam row too: out of 100 spam messages, the cautious filter misses 25, the lessons filter 10 (its **recall** is 0.90 versus 0.75). The right choice depends on what costs more: an important message lost in “Junk”, or one more spam in the inbox. As in chapter 3, you must decide on the measure before comparing. The curve shows that the gap between the two filters is largest when spam is rare.

## Exercise 4.3 — Three spam messages, then ten messages

*Concepts: Bernoulli trial and binomial distribution (lesson 4.5), conditional probability (lesson 4.3).* First do the questions on paper.

### a) to e) Solution with comments

**a)** Each branch multiplies 0.9 (flagged) or 0.1 (missed). The outcome “three flagged” is $0{,}9^3 = 0{,}729$; each outcome with a single missed spam is $0{,}1 \times 0{,}9^2 = 0{,}081$; with two missed, $0{,}1^2 \times 0{,}9 = 0{,}009$; all three missed, $0{,}1^3 = 0{,}001$.

**b)** Three outcomes have a single missed spam: $P(X = 1) = 3 \times 0{,}081 = 0{,}243$. Likewise $P(X = 2) = 3 \times 0{,}009 = 0{,}027$, $P(X = 0) = 0{,}729$, $P(X = 3) = 0{,}001$. Sum: $0{,}729 + 0{,}243 + 0{,}027 + 0{,}001 = 1$. The factor 3 is the coefficient $\binom{3}{1} = \binom{3}{2} = 3$.

**c)** By adding: $0{,}243 + 0{,}027 + 0{,}001 = 0{,}271$. Using the complement: $1 - 0{,}729 = 0{,}271$. The second calculation is shorter.

**d)** **Check which set you are counting in.** We no longer count over ten *spam messages* but over ten *received messages*, some of which are legitimate. For a received message, “spam that gets through” means “spam **and** not flagged”: $0{,}2 \times 0{,}1 = 0{,}02$ (lesson 4.3). None of the ten gets through with probability $0{,}98^{10} \approx 0{,}817$, so at least one spam gets through with probability $1 - 0{,}98^{10} \approx 0{,}183$.

**e)** In lesson 4.5, the ten messages are all spam: each has a 10 % chance of getting through. Among ten received messages, there are only two spam messages on average, and each message has only a 2 % chance of being a spam that gets through. “Ten spam messages” and “ten received messages” are two different questions.

In [ ]:
loi_3 = [comb(3, k) * 0.1 ** k * 0.9 ** (3 - k) for k in range(4)]   # over 3 SPAM messages
p_d = 1 - (1 - 0.2 * 0.1) ** 10   # over 10 RECEIVED MESSAGES: spam AND not flagged = 0.02

In [ ]:
# Check
if loi_3 is None or p_d is None:
    print('Complete the previous cell.')
else:
    print('b):', [round(v, 3) for v in loi_3], '→', 'correct' if np.allclose(loi_3, [0.729, 0.243, 0.027, 0.001]) else 'check again')
    print(f'd): {p_d:.4f} →', 'correct' if abs(p_d - 0.1829) < 0.0001 else 'check again')

## Exercise 4.4 — Simulating ten spam messages

*Concepts: binomial distribution (lesson 4.5), frequency and probability.* We count over **ten spam messages**.

In [ ]:
def simuler_serie(rng):
    """Returns the number of missed spam messages out of 10. rng.random() draws a number between 0 and 1."""
    return int(np.sum(rng.random(10) < 0.1))   # one draw per spam: missed if < 0.1

In [ ]:
# b) and c) 10 000 series (fixed seed to get the same numbers)
rng = np.random.default_rng(4)
if simuler_serie(rng) is None:
    print('Complete the function.')
else:
    rng = np.random.default_rng(4)
    X = np.array([simuler_serie(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'observed frequency: {freq:.4f}; calculated probability: {1 - 0.9 ** 10:.4f} →',
          'correct' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'check again')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=SIGNALE, label='observed frequency')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='probability (equation 4.7)')
    plt.xlabel('Missed spam messages out of 10'); plt.legend(); plt.show()
    # d) More series: the frequency gets closer to the probability
    for N in (100, 100, 100, 100000):
        Y = np.array([simuler_serie(rng) for _ in range(N)])
        print(f'{N:>6} series: frequency = {np.mean(Y >= 1):.4f}')

### Solution with comments

With the notebook's seed, 10 000 series give a frequency of 0.6549, for a probability of 0.6513. The two are close, but not equal: an observed frequency varies from one simulation to the next. With 100 series, it can be off by several percentage points (rerun the cell without a seed to see it). With 100 000 series, it gets within a few thousandths. The probability is the value the frequency tends towards as the number of series grows. Also check the set: each series counts **spam messages**, never received messages.

## Exercise 4.5 — Predict, then check: a spam campaign

*Concepts: independence, complementary event (lesson 4.5).*

### a) to c) and e) Solution with comments

**b)** The filter misses all ten spam messages at once with probability 0.1, or misses none. So the number of missed spam messages is 0 or 10. At least one gets through with probability **0.1**, far less than 65 %.

**c)** On average: $10 \times 0{,}1 + 0 \times 0{,}9 = 1$ missed spam. It is the same mean as in lesson 4.5 ($10 \times 0{,}1 = 1$). The mean is identical, but the distribution is very different.

**e)** Independence no longer holds: knowing that the first spam is missed makes the others almost certainly missed. The probability of being missed, 0.1 for each, has not changed. For the user, ten spam messages at once are more annoying, and more visible. The binomial distribution only applies when both of its assumptions hold.

In [ ]:
def simuler_campagne(rng):
    return 10 if rng.random() < 0.1 else 0   # a single draw for the whole campaign

In [ ]:
# d) Check by simulation
rng = np.random.default_rng(5)
if simuler_campagne(rng) is None:
    print('Complete the function.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([simuler_campagne(rng) for _ in range(10000)])
    print(f'at least one missed: {np.mean(Z >= 1):.4f}; mean: {Z.mean():.3f} →',
          'correct' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'check again')

## Exercise 4.6 — Challenge: how many spam messages before the risk of missing one exceeds 99 %?

*Concepts: Bernoulli and binomial distributions, normal distribution (lesson 4.5), standard deviation (lesson 1.1).*

### a) to d) Solution with comments

**a)** Let $X_i = 1$ if spam $i$ is missed, 0 otherwise. Then $X = X_1 + \dots + X_n$: adding up the 1s counts the missed spam messages. Each $X_i$ has mean $1 \times p + 0 \times (1 - p) = p$, and the mean of a sum is the sum of the means: $n\,p$. For 10 spam messages, 1.

**b)** We need $1 - 0{,}9^n > 0{,}99$, that is $0{,}9^n < 0{,}01$. We find $0{,}9^{43} \approx 0{,}0108$ and $0{,}9^{44} \approx 0{,}0097$: it takes **44 spam messages**.

**c)** Mean: $100 \times 0{,}1 = 10$. Standard deviation: $\sqrt{100 \times 0{,}1 \times 0{,}9} = \sqrt{9} = 3$.

**d)** The bars of the binomial distribution almost hug the bell curve with mean 10 and standard deviation 3. When $n$ is large, the binomial distribution looks like a normal distribution: that is why the curve from lesson 4.5 will come back often.

In [ ]:
n_min = next(n for n in range(1, 200) if 0.9 ** n < 0.01)   # we try n = 1, 2, 3…
moyenne_100 = 100 * 0.1
ecart_type_100 = np.sqrt(100 * 0.1 * 0.9)

In [ ]:
# Check, then d) binomial versus normal distribution
if None in (n_min, moyenne_100, ecart_type_100):
    print('Complete the previous cell.')
else:
    print(f'n_min = {n_min} →', 'correct' if n_min == 44 else 'check again')
    print(f'mean = {moyenne_100:.1f}, standard deviation = {ecart_type_100:.1f} →',
          'correct' if np.isclose(moyenne_100, 10) and np.isclose(ecart_type_100, 3) else 'check again')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='binomial distribution (n = 100, p = 0.1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='normal distribution (mean 10, standard deviation 3)')
    plt.xlabel('Missed spam messages out of 100'); plt.legend(); plt.show()